In [ ]:
*Title*: 2_Allele_Frequency_Differences_and_Outliers
*Goal*: Generate files for either outlier SNPs (based on FST) and/or neutral SNPs. Also, generate allele frequency difference files for 2016 and 2017.
Based on original script: PCA_outliers

# 2.1 Generate allele frequency differences  ----------------------
Showing for 2016 only, but 2017 same

``` bash
module load StdEnv/2020
perl popoolation2_1201/snp-frequency-diff.pl --input 1.2016masked-autosome-chrom-reorder.sync --output-prefix 5.2016masked-autosome-allele-freq-diff  --min-count 2 --min-coverage 5 --max-coverage 100 
``` 
# 2.2 Restructure the file  ----------------------
The  output file lists SNPs that violate the filtering parameters as 0 (i.e. coverage > 100, the output file recomputes it to be 0/101) instead of removing the line altogether! So removing lines that violate filtering parameters. Also rearranging the file to match the structure of other files.

``` R
library("tidyverse")

#2016 file generation:
allele_freq_data2016 <- read.table("5.2016masked-autosome-allele-freq-diff_rc", header = FALSE) %>%
  mutate(Chromosome_Position = paste(V1, V2, sep = "_")) %>%   # Combine 'chr' and 'pos' columns into 'Chromosome_Position'
  select(Chromosome_Position, V10:V21) #I want to select the major allele columns 

colnames(allele_freq_data2016)<- c("Chromosome_Position", "Younger_Spring-2016","Younger_Fall-2016","OldDairy_Spring-2016","OldDairy_Fall-2016","Scott_Spring-2016","Scott_Fall-2016",
"Lombardi_Spring-2016","Lombardi_Fall-2016","Laguna_Spring-2016","Laguna_Fall-2016","Waddell_Spring-2016","Waddell_Fall-2016")

write.table(allele_freq_data2016,"allele_freq_data_2016.tsv", sep = "\t",row.names=FALSE, quote =FALSE)
``` 
# 2.3 Filtering ----------------------
Generate 2016 and 2017 files that are either neutral (no outliers of that year) OR outliers
outlier file will show how similar/different selection is across populations
neutral will show population structure
Filtering both files to only include relevant SNPs:

Keep ALL SNPs present in both years (whether neutral or outliers):
```bash
  #First I have to get the column names, but I dont want the first column (contains population names):
  #The output file is reorganized so that the columns are now rows in 1 column called chromosome_position:

  #Now filter each file to only include the Snps:
  wc -l allele_freq_data_2016.tsv #9020067 Snps (representing alleles from 2016 populations assessed for major alleles)
 awk 'FILENAME=="all_snps.tsv" {a[$1]; next} $1 in a' all_snps.tsv allele_freq_data_2016.tsv > allele_freq_data_2016_neutral_and_outliers.tsv 
 
  wc -l allele_freq_data_2016_neutral_and_outliers.tsv 
  
  wc -l allele_freq_data_2017.tsv #=24553592
  awk 'FILENAME=="all_snps.tsv" {a[$1]; next} $1 in a' all_snps.tsv allele_freq_data_2017.tsv > allele_freq_data_2017_neutral_and_outliers.tsv 
  wc -l allele_freq_data_2017_neutral_and_outliers.tsv 

#Note that the wc-l are not exactly the same. Since all_snps was based off the poolfstat fst outliers and the all_freq is from the popoolation allele_freq, it is possible that both programs have slightly different results despite the same filtering parameters.

# Outliers Only, Neutral Only
#2016 outliers
  #filtering to only include the outliers
  awk 'FILENAME=="2016_outliers.tsv" {a[$1]; next} $1 in a' 2016_outliers.tsv allele_freq_data_2016_neutral_and_outliers.tsv > allele_freq_data_2016_outliers.tsv
  wc -l allele_freq_data_2016_outliers.tsv # 1572903 Snps

  
#2016 neutral
  #filtering to NOT include the outliers
(head -n 1 allele_freq_data_2016_neutral_and_outliers.tsv && awk 'FILENAME=="2016_outliers.tsv" {a[$1]; next} NR>1 && !($1 in a)' 2016_outliers.tsv allele_freq_data_2016_neutral_and_outliers.tsv) > allele_freq_data_2016_neutral.tsv


  #awk 'FILENAME=="2016_outliers.tsv" {a[$1]; next} !($1 in a)' 2016_outliers.tsv allele_freq_data_2016_neutral_and_outliers.tsv > allele_freq_data_2016_neutral.tsv
  wc -l allele_freq_data_2016_neutral.tsv 
  ```

# 2.4 Cleaning the file  ---------------------- 
*Goal:* Clean all files made above using a loop. Separate versions for both years, as they both have different filtering restrictions. 
*OPTIONAL* Please note that this is a sanity check to ensure that no SNPs violate filtering parameters and/or are not numerical. Please note that no outlier loci (i.e., those graphed in PCAs) violated the sanity check.

2017 version:
  ``` R
library(data.table)
check_greater_parameters <- function(row) {
  results <- sapply(row, function(cell) {
    parts <- strsplit(as.character(cell), "/")[[1]]
    if(length(parts) < 2) return(FALSE)
    num <- as.numeric(parts[1])
    denom <- as.numeric(parts[2])
    denom < 5 || denom > 300
  })
  any(results)
}

file_list <- list.files(pattern = "allele_freq_data_2017.*\\.tsv")

#Loop through each file
for (file in file_list) {
  #Read the data with column and row names
  allele_freq_data <- read.table(file, header = TRUE, sep = "\t", stringsAsFactors = FALSE, row.names = 1)

  #Check for NA before alterations
  print(paste("Before alteration for file:", file))
  base_name <- sub("\\.tsv$", "", basename(file))
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
  print(nrow(allele_freq_data))

  #Remove rows that didn't pass max coverage limit
  allele_freq_data <- allele_freq_data[!apply(allele_freq_data, 1, check_greater_parameters), ]

  print("after coverage filtering")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
   print(nrow(allele_freq_data))

  #Fraction conversion
row_names <- row.names(allele_freq_data)

allele_freq_data <- as.data.frame(lapply(allele_freq_data, function(x) {
  sapply(strsplit(as.character(x), "/"), function(y) as.numeric(y[1]) / as.numeric(y[2]))
}))

#Reassign the row names
row.names(allele_freq_data) <- row_names
  print("after convert from fraction")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
 print("number of SNPS:")
   print(nrow(allele_freq_data))


  #Transpose the data to have samples as rows and SNPs as columns
  allele_freq_data <- as.data.frame(t(allele_freq_data))
  print("after transposition:")
  #The the snps are now the columns
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))

#To look at number of constants removed:
print("number of snps before constants removed")
print(ncol(allele_freq_data))

  #Identify and remove constant/zero-variance columns
  constant_cols <- apply(allele_freq_data, 2, function(x) length(unique(x)) == 1)
  constant_cols_indices <- which(constant_cols)
  allele_freq_data <- allele_freq_data[, !constant_cols]
  constant_cols_data <- data.frame(Column_Index = constant_cols_indices)
  write.table(as.data.frame(constant_cols_data), paste0(base_name, "_constant_cols.tsv"), sep = "\t", row.names = TRUE, col.names = TRUE, quote = FALSE)
  print("number of snps after constants removed")
print(ncol(allele_freq_data))
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))
  print(sum(is.na(allele_freq_data)))
allele_freq_data$population <- row.names(allele_freq_data)
allele_freq_data <- allele_freq_data[, c("population", setdiff(names(allele_freq_data), "population"))] #doing this so i can have the pop names saved

fwrite(allele_freq_data, paste0(base_name, "_no_constants.tsv"), sep = "\t", quote = FALSE)
print("done")
}
 ```

2016 version:
``` R
library(data.table)

check_greater_parameters <- function(row) {
  results <- sapply(row, function(cell) {
    parts <- strsplit(as.character(cell), "/")[[1]]
    if(length(parts) < 2) return(FALSE)
    num <- as.numeric(parts[1])
    denom <- as.numeric(parts[2])
    denom < 5 || denom > 100
  })
  any(results)
}

file_list <- list.files(pattern = "allele_freq_data_2016.*\\.tsv")

# Loop through each file
for (file in file_list) {
  # Read the data with column and row names
  allele_freq_data <- read.table(file, header = TRUE, sep = "\t", stringsAsFactors = FALSE, row.names = 1)

  # Check for NA before alterations
  print(paste("Before alteration for file:", file))
  base_name <- sub("\\.tsv$", "", basename(file))
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
  print(nrow(allele_freq_data))

  # Remove rows that didn't pass max coverage limit
  allele_freq_data <- allele_freq_data[!apply(allele_freq_data, 1, check_greater_parameters), ]

  print("after coverage filtering")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
  print("number of SNPS:")
   print(nrow(allele_freq_data))

  # Fraction conversion
row_names <- row.names(allele_freq_data)

allele_freq_data <- as.data.frame(lapply(allele_freq_data, function(x) {
  sapply(strsplit(as.character(x), "/"), function(y) as.numeric(y[1]) / as.numeric(y[2]))
}))

# Reassign the row names
row.names(allele_freq_data) <- row_names
  print("after convert from fraction")
  print("number of NA")
  print(sum(is.na(allele_freq_data)))
 print("number of SNPS:")
   print(nrow(allele_freq_data))

  print("after transposition:")

  # Transpose the data to have samples as rows and SNPs as columns
  allele_freq_data <- as.data.frame(t(allele_freq_data))

  #The the snps are now the columns
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))

# To look at number of constants removed:
print("number of snps before constants removed")
print(ncol(allele_freq_data))

  # Identify and remove constant/zero-variance columns
  constant_cols <- apply(allele_freq_data, 2, function(x) length(unique(x)) == 1)
  constant_cols_indices <- which(constant_cols)
  allele_freq_data <- allele_freq_data[, !constant_cols]
  constant_cols_data <- data.frame(Column_Index = constant_cols_indices)
  write.table(as.data.frame(constant_cols_data), paste0(base_name, "_constant_cols.tsv"), sep = "\t", row.names = TRUE, col.names = TRUE, quote = FALSE)
  print("number of snps after constants removed")
print(ncol(allele_freq_data))
  print(allele_freq_data[1:2, 1:2])
  print(class(allele_freq_data))
  print(sum(is.na(allele_freq_data)))
allele_freq_data$population <- row.names(allele_freq_data)
allele_freq_data <- allele_freq_data[, c("population", setdiff(names(allele_freq_data), "population"))] #doing this so i can have the pop names saved

fwrite(allele_freq_data, paste0(base_name, "_no_constants.tsv"), sep = "\t", quote = FALSE)
print("done")
}
```